# PLACES Dataset Cleaning
Haylee Oyler
7/11/2026


In [1]:
# Load packages 
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import os

# Remove max columns
pd.set_option('display.max_columns', None)


In [2]:
# Import data
# base_dir = "/capstone/justice40"
base_dir = "~/Jobs/CCREI/network-graphs/data"

# Census tract level PLACES data
places_raw = pd.read_csv(os.path.join(base_dir,"PLACES__Local_Data_for_Better_Health,_Census_Tract_Data,_2025_release_20260711.csv"))

# Data dictionary
data_dict = pd.read_csv(os.path.join(base_dir, "PLACES_and_500_Cities__Data_Dictionary_20260711.csv"))

# GEOJSON version census tract data
places_geo_raw = pd.read_csv(os.path.join(base_dir, "PLACES__Census_Tract_Data_(GIS_Friendly_Format),_2025_release_20260711.geojson"))


/var/folders/ck/21jw_cxj7j984t_wcgbyrv200000gn/T/ipykernel_71998/653982250.py:12: DtypeWarning: Columns (128,129,130,131,132,133,134) have mixed types. Specify dtype option on import or set low_memory=False.
  places_geo_raw = pd.read_csv(os.path.join(base_dir, "PLACES__Census_Tract_Data_(GIS_Friendly_Format),_2025_release_20260711.geojson"))


In [3]:
places_raw.head()

,Year,StateAbbr,StateDesc,CountyName,CountyFIPS,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,Data_Value_Footnote_Symbol,Data_Value_Footnote,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,Geolocation,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text
0,2023,CA,California,San Luis Obispo,6079,6079010603,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,4.3,NaN,NaN,3.7,4.9,"1,407","1,230",POINT (-120.8299539 35.3722416),6079010603,HLTHOUT,STROKE,CrdPrv,Stroke
1,2023,CA,California,Santa Barbara,6083,6083000501,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,3.0,NaN,NaN,2.7,3.4,"3,328","2,840",POINT (-119.7080007 34.4674371),6083000501,HLTHOUT,STROKE,CrdPrv,Stroke
2,2023,CA,California,Santa Barbara,6083,6083000806,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,2.9,NaN,NaN,2.6,3.3,"5,689","4,347",POINT (-119.6764178 34.4232883),6083000806,HLTHOUT,STROKE,CrdPrv,Stroke
3,2023,CA,California,Santa Barbara,6083,6083001002,BRFSS,Health Outcomes,Obesity among adults,%,Crude prevalence,27.9,NaN,NaN,22.5,33.9,"3,104","2,727",POINT (-119.7092125 34.4210021),6083001002,HLTHOUT,OBESITY,CrdPrv,Obesity
4,2023,CA,California,Santa Barbara,6083,6083001907,BRFSS,Health Outcomes,Obesity among adults,%,Crude prevalence,26.6,NaN,NaN,21.1,32.5,940,745,POINT (-120.0082874 34.5499261),6083001907,HLTHOUT,OBESITY,CrdPrv,Obesity


In [4]:
places_raw.shape

(14480, 24)

In [5]:
print(np.unique(places_raw['Year']))
print(np.unique(places_raw['Category']))
print(np.unique(places_raw['Short_Question_Text']))

[2022 2023]
['Disability' 'Health Outcomes' 'Health Risk Behaviors' 'Health Status'
 'Health-Related Social Needs' 'Prevention']
['All Teeth Lost' 'Annual Checkup' 'Any Disability' 'Arthritis'
 'Binge Drinking' 'COPD' 'Cancer (non-skin) or Melanoma'
 'Cholesterol Screening' 'Cognitive Disability'
 'Colorectal Cancer Screening' 'Coronary Heart Disease' 'Current Asthma'
 'Current Cigarette Smoking' 'Dental Visit' 'Depression' 'Diabetes'
 'Food Insecurity' 'Food Stamps' 'Frequent Mental Distress'
 'Frequent Physical Distress' 'General Health' 'Health Insurance'
 'Hearing Disability' 'High Blood Pressure'
 'High Blood Pressure Medication' 'High Cholesterol' 'Housing Insecurity'
 'Independent Living Disability' 'Lack of Social/Emotional Support'
 'Loneliness' 'Mammography' 'Mobility Disability' 'Obesity'
 'Physical Inactivity' 'Self-care Disability' 'Short Sleep Duration'
 'Stroke' 'Transportation Barriers' 'Utility Services Threat'
 'Vision Disability']


In [21]:
insurance = places_raw[places_raw['Short_Question_Text'] == 'Health Insurance']
insurance.shape

(362, 24)

In [22]:
insurance.head(6)

,Year,StateAbbr,StateDesc,CountyName,CountyFIPS,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,Data_Value_Footnote_Symbol,Data_Value_Footnote,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,Geolocation,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text
39,2023,CA,California,Santa Barbara,6083,6083002012,BRFSS,Prevention,Current lack of health insurance among adults ...,%,Crude prevalence,7.7,NaN,NaN,6.1,9.9,"3,209","2,446",POINT (-120.4436472 34.8686385),6083002012,PREVENT,ACCESS2,CrdPrv,Health Insurance
53,2023,CA,California,San Luis Obispo,6079,6079010202,BRFSS,Prevention,Current lack of health insurance among adults ...,%,Crude prevalence,10.9,NaN,NaN,8.5,13.6,"5,499","4,211",POINT (-120.6302235 35.5983407),6079010202,PREVENT,ACCESS2,CrdPrv,Health Insurance
72,2023,CA,California,San Luis Obispo,6079,6079012102,BRFSS,Prevention,Current lack of health insurance among adults ...,%,Crude prevalence,9.3,NaN,NaN,7.2,11.6,"5,482","4,459",POINT (-120.6245239 35.1222342),6079012102,PREVENT,ACCESS2,CrdPrv,Health Insurance
81,2023,CA,California,Santa Barbara,6083,6083000801,BRFSS,Prevention,Current lack of health insurance among adults ...,%,Crude prevalence,15.5,NaN,NaN,11.5,20.0,"4,021","3,140",POINT (-119.6856662 34.4273594),6083000801,PREVENT,ACCESS2,CrdPrv,Health Insurance
191,2023,CA,California,San Luis Obispo,6079,6079012406,BRFSS,Prevention,Current lack of health insurance among adults ...,%,Crude prevalence,13.4,NaN,NaN,10.3,16.6,"3,677","2,706",POINT (-120.4792384 35.0441765),6079012406,PREVENT,ACCESS2,CrdPrv,Health Insurance
311,2023,CA,California,San Luis Obispo,6079,6079010404,BRFSS,Prevention,Current lack of health insurance among adults ...,%,Crude prevalence,7.3,NaN,NaN,5.5,9.4,"2,219","1,915",POINT (-121.0948278 35.5748544),6079010404,PREVENT,ACCESS2,CrdPrv,Health Insurance
